# ICT4403 — Foundations of Generative AI
## Assignment 2: Autoregressive Text Generation with RNN, BiRNN, LSTM and GRU

**Framework:** PyTorch  
**Dataset:** Tiny Shakespeare (downloaded automatically)

### How to execute
1. Upload this notebook to Google Colab.
2. Choose **Runtime → Run all**.
3. Wait for the four models to train. At the end, the notebook creates and downloads `ICT4403_Assignment2_Results.zip` automatically.

No code edits are required. This lightweight configuration uses 12,000 tokens and 2 epochs per model so it can finish faster. Results and plots are generated from the actual run; they are not prefilled or fabricated.

## 1. Experimental design

To keep the comparison fair, the models use the same tokenized dataset, train/validation/test split, context length, batch size, optimizer, learning rate, epoch limit, and evaluation procedure. The model family is the main architectural difference. The bidirectional model sees both directions **within the supplied context only**; it does not receive the target word or any future words.

Metrics reported:
- **Cross-entropy loss**: lower is better.
- **Perplexity**: `exp(loss)`; lower is better.
- **Next-word accuracy**: higher is better.
- **Training time**: wall-clock time for training.
- **Qualitative review**: coherence, diversity, contextual consistency, and repetition in generated text.

The dataset is deliberately capped by `MAX_TOKENS` to make experiments practical on a laptop. You can increase it if you have sufficient compute.

In [ ]:
import os, re, time, math, random, urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Download and inspect the dataset

Tiny Shakespeare is a compact corpus of Shakespeare's works commonly used for language-modeling demonstrations. The notebook tokenizes words and punctuation separately, making the prediction target the **next token/word-like unit**. Punctuation is retained as tokens so generated text can form sentence-like sequences.

In [ ]:
# Automatically detect the project folder. In Google Colab, files are saved under /content.
CWD = Path.cwd()
if CWD.name == "notebooks":
    PROJECT_DIR = CWD.parent
elif (CWD / "notebooks").exists() or (CWD / "README.md").exists():
    PROJECT_DIR = CWD
else:
    PROJECT_DIR = CWD

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = DATA_DIR / "tinyshakespeare.txt"

if not DATA_PATH.exists():
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

raw_text = DATA_PATH.read_text(encoding="utf-8")
print("Characters:", len(raw_text))
print("Preview:\n", raw_text[:500])

## 3. Preprocess text and build next-token sequences

Preprocessing:
1. Normalize repeated whitespace.
2. Split words and punctuation using a regular expression.
3. Split the token stream chronologically into train, validation, and test partitions.
4. Build the vocabulary from the training partition only to avoid leakage.
5. Replace out-of-vocabulary validation/test tokens with `<unk>`.
6. Create sliding context-target pairs: a context of `CONTEXT_LEN` tokens predicts the immediately following token.

In [ ]:
# Lightweight settings for a quick, comparable run in Google Colab.
MAX_TOKENS = 12000
CONTEXT_LEN = 12
BATCH_SIZE = 128
EMBED_DIM = 96
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.15
LEARNING_RATE = 2e-3
EPOCHS = 2
MAX_VOCAB = 8000
NUM_WORKERS = 0

# Keep contractions and words together; punctuation is emitted as separate tokens.
tokens_all = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\w\s]", raw_text.lower())
tokens_all = tokens_all[:MAX_TOKENS]
print("Token count used:", len(tokens_all))
print("Sample tokens:", tokens_all[:40])

n = len(tokens_all)
train_end = int(n * 0.80)
val_end = int(n * 0.90)
train_tokens = tokens_all[:train_end]
val_tokens = tokens_all[train_end:val_end]
test_tokens = tokens_all[val_end:]

counts = Counter(train_tokens)
specials = ["<pad>", "<unk>"]
vocab_words = [w for w, _ in counts.most_common(MAX_VOCAB - len(specials)) if w not in specials]
itos = specials + vocab_words
stoi = {w:i for i,w in enumerate(itos)}
PAD_IDX, UNK_IDX = stoi["<pad>"], stoi["<unk>"]

def encode(tokens):
    return [stoi.get(t, UNK_IDX) for t in tokens]

train_ids = encode(train_tokens)
val_ids = encode(val_tokens)
test_ids = encode(test_tokens)

print(f"Vocabulary size: {len(itos):,}")
print(f"Train/validation/test tokens: {len(train_ids):,} / {len(val_ids):,} / {len(test_ids):,}")

In [ ]:
class NextTokenDataset(Dataset):
    def __init__(self, token_ids, context_len):
        self.ids = torch.tensor(token_ids, dtype=torch.long)
        self.context_len = context_len
    def __len__(self):
        return max(0, len(self.ids) - self.context_len)
    def __getitem__(self, idx):
        x = self.ids[idx:idx+self.context_len]
        y = self.ids[idx+self.context_len]
        return x, y

train_ds = NextTokenDataset(train_ids, CONTEXT_LEN)
val_ds = NextTokenDataset(val_ids, CONTEXT_LEN)
test_ds = NextTokenDataset(test_ids, CONTEXT_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Examples:", len(train_ds), len(val_ds), len(test_ds))
x_example, y_example = train_ds[0]
print("Input context:", [itos[i] for i in x_example.tolist()])
print("Target next token:", itos[y_example.item()])

## 4. Define the four model architectures

All models embed the context tokens, process the sequence, and classify the final representation into the vocabulary. For the bidirectional RNN, the final forward and backward hidden states are concatenated. Each model predicts one next token from the same fixed-length context.

In [ ]:
class NextTokenModel(nn.Module):
    def __init__(self, architecture, vocab_size, embed_dim, hidden_dim,
                 num_layers=1, dropout=0.1):
        super().__init__()
        self.architecture = architecture.lower()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        recurrent_dropout = dropout if num_layers > 1 else 0.0

        if self.architecture == "rnn":
            self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers=num_layers,
                              batch_first=True, nonlinearity="tanh",
                              dropout=recurrent_dropout)
            out_dim = hidden_dim
        elif self.architecture == "birnn":
            self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers=num_layers,
                              batch_first=True, bidirectional=True,
                              nonlinearity="tanh", dropout=recurrent_dropout)
            out_dim = hidden_dim * 2
        elif self.architecture == "lstm":
            self.rnn = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                               batch_first=True, dropout=recurrent_dropout)
            out_dim = hidden_dim
        elif self.architecture == "gru":
            self.rnn = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers,
                              batch_first=True, dropout=recurrent_dropout)
            out_dim = hidden_dim
        else:
            raise ValueError("architecture must be rnn, birnn, lstm, or gru")

        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(out_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, state = self.rnn(embedded)
        if self.architecture == "birnn":
            # state is hidden state for bidirectional RNN: (layers*2, batch, hidden)
            h = state[-2:].transpose(0, 1).reshape(x.size(0), -1)
        elif self.architecture == "lstm":
            h_n, c_n = state
            h = h_n[-1]
        else:
            h = state[-1]
        return self.head(self.dropout(h))

ARCHITECTURES = ["rnn", "birnn", "lstm", "gru"]

## 5. Train and evaluate consistently

Training uses Adam and cross-entropy loss. Validation loss selects the best checkpoint for each model. The test split is held back until training is complete. Perplexity is computed as the exponential of average cross-entropy; if loss is extremely large, perplexity may overflow and is reported as infinity.

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            batch_n = yb.size(0)
            total_loss += loss.item() * batch_n
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total += batch_n
    avg_loss = total_loss / max(total, 1)
    try:
        ppl = math.exp(avg_loss)
    except OverflowError:
        ppl = float("inf")
    return {"loss": avg_loss, "perplexity": ppl,
            "accuracy": total_correct / max(total, 1)}

def train_one(architecture):
    model = NextTokenModel(architecture, len(itos), EMBED_DIM, HIDDEN_DIM,
                           NUM_LAYERS, DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    best_val_loss = float("inf")
    best_state = None
    history = []
    start = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, seen = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * yb.size(0)
            seen += yb.size(0)

        train_loss = running_loss / max(seen, 1)
        val_metrics = evaluate(model, val_loader)
        history.append({"epoch": epoch, "train_loss": train_loss,
                        "val_loss": val_metrics["loss"],
                        "val_perplexity": val_metrics["perplexity"],
                        "val_accuracy": val_metrics["accuracy"]})
        print(f"{architecture.upper():5s} | epoch {epoch}/{EPOCHS} "
              f"| train loss {train_loss:.4f} | val loss {val_metrics['loss']:.4f} "
              f"| val acc {val_metrics['accuracy']:.4f}")

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    elapsed = time.perf_counter() - start
    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(DEVICE)
    return model, pd.DataFrame(history), elapsed

models, histories, train_times = {}, {}, {}
for arch in ARCHITECTURES:
    print("\\n" + "="*20, arch.upper(), "="*20)
    model, hist, elapsed = train_one(arch)
    models[arch], histories[arch], train_times[arch] = model, hist, elapsed
    torch.save(model.state_dict(), OUTPUT_DIR / f"{arch}_best.pt")
    hist.to_csv(OUTPUT_DIR / f"{arch}_history.csv", index=False)
    print(f"Training wall time: {elapsed:.1f} seconds")

## 6. Test-set quantitative evaluation

The test metrics are computed using the checkpoint with the lowest validation loss for each architecture. Results are saved as a CSV for the report and README.

In [ ]:
rows = []
for arch, model in models.items():
    metrics = evaluate(model, test_loader)
    rows.append({
        "model": arch.upper(),
        "test_loss": metrics["loss"],
        "test_perplexity": metrics["perplexity"],
        "test_accuracy": metrics["accuracy"],
        "training_time_seconds": train_times[arch]
    })
results = pd.DataFrame(rows).sort_values("test_loss").reset_index(drop=True)
results.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
display(results)

## 7. Plot learning curves and model comparison

These figures are generated from the actual run and saved under `outputs/`. Use them in your submitted analysis.

In [ ]:
# Training and validation loss curves
fig, ax = plt.subplots(figsize=(10, 6))
for arch, hist in histories.items():
    ax.plot(hist["epoch"], hist["train_loss"], marker="o", label=f"{arch.upper()} train")
    ax.plot(hist["epoch"], hist["val_loss"], marker=".", linestyle="--", label=f"{arch.upper()} validation")
ax.set(title="Training and Validation Loss by Epoch", xlabel="Epoch", ylabel="Cross-entropy loss")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "learning_curves.png", dpi=160)
plt.show()

metrics_to_plot = [
    ("test_loss", "Test Cross-Entropy Loss (lower is better)", "model_comparison_loss.png"),
    ("test_perplexity", "Test Perplexity (lower is better)", "model_comparison_perplexity.png"),
    ("test_accuracy", "Test Next-Token Accuracy (higher is better)", "model_comparison_accuracy.png"),
    ("training_time_seconds", "Training Time in Seconds (lower is faster)", "model_comparison_training_time.png"),
]
for col, title, filename in metrics_to_plot:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(results["model"], results[col])
    ax.set(title=title, xlabel="Model", ylabel=col.replace("_", " ").title())
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=160)
    plt.show()

## 8. Autoregressive generation at different temperatures

Temperature rescales the next-token logits before softmax:
- **T < 1** sharpens the distribution, usually making outputs more predictable and repetitive.
- **T = 1** uses the model's learned distribution without temperature scaling.
- **T > 1** flattens the distribution, often increasing diversity but also errors and incoherence.

The exact effects must be judged from the generated examples rather than assumed. All models use the same seed text and temperatures.

In [ ]:
def sample_next_token(logits, temperature=1.0, top_k=40):
    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    logits = logits / temperature
    if top_k is not None and top_k > 0:
        k = min(top_k, logits.numel())
        values, indices = torch.topk(logits, k)
        filtered = torch.full_like(logits, float("-inf"))
        filtered[indices] = values
        logits = filtered
    probs = torch.softmax(logits, dim=-1)
    return int(torch.multinomial(probs, num_samples=1).item())

def generate(model, seed_text, temperature=1.0, max_new_tokens=80, top_k=40):
    seed_tokens = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\w\s]", seed_text.lower())
    if not seed_tokens:
        raise ValueError("Seed text must contain at least one token.")
    ids = [stoi.get(t, UNK_IDX) for t in seed_tokens]
    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = ids[-CONTEXT_LEN:]
            if len(context) < CONTEXT_LEN:
                context = [PAD_IDX] * (CONTEXT_LEN - len(context)) + context
            xb = torch.tensor([context], dtype=torch.long, device=DEVICE)
            logits = model(xb)[0]
            next_id = sample_next_token(logits, temperature, top_k)
            ids.append(next_id)
    generated_tokens = [itos[i] for i in ids]
    # Simple detokenization: punctuation attaches to preceding token.
    out = ""
    no_space_before = {".", ",", "!", "?", ";", ":", ")", "]", "}", "'s"}
    no_space_after = {"(", "[", "{"}
    for tok in generated_tokens:
        if not out:
            out = tok
        elif tok in no_space_before or out[-1:] in no_space_after:
            out += tok
        else:
            out += " " + tok
    return out

SEED_TEXTS = [
    "to be or not to be",
    "the king said",
]
TEMPERATURES = [0.5, 1.0, 1.3]

generation_rows = []
for arch, model in models.items():
    for seed_text in SEED_TEXTS:
        for temp in TEMPERATURES:
            generated = generate(model, seed_text, temperature=temp, max_new_tokens=80, top_k=40)
            generation_rows.append({
                "model": arch.upper(),
                "seed": seed_text,
                "temperature": temp,
                "generated_text": generated
            })
            print(f"\\n### {arch.upper()} | seed={seed_text!r} | temperature={temp}")
            print(generated)

generation_df = pd.DataFrame(generation_rows)
generation_df.to_csv(OUTPUT_DIR / "generated_samples.csv", index=False)

## 9. Lightweight repetition and diversity diagnostics

These simple diagnostics do not replace human evaluation. `unique_token_ratio` approximates lexical diversity, while `repeated_bigram_ratio` measures how often generated bigrams repeat. Review samples manually for coherence, contextual consistency, and whether the model collapses into repeated phrases.

In [ ]:
def generation_diagnostics(text):
    toks = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\w\s]", str(text).lower())
    lexical = [t for t in toks if re.match(r"\w", t)]
    bigrams = list(zip(toks, toks[1:]))
    unique_ratio = len(set(lexical)) / max(len(lexical), 1)
    repeated_bigram_ratio = 1 - len(set(bigrams)) / max(len(bigrams), 1)
    return pd.Series({
        "token_count": len(toks),
        "unique_token_ratio": unique_ratio,
        "repeated_bigram_ratio": repeated_bigram_ratio
    })

diag = generation_df["generated_text"].apply(generation_diagnostics)
generation_analysis = pd.concat([generation_df.drop(columns=["generated_text"]), diag], axis=1)
generation_analysis.to_csv(OUTPUT_DIR / "generation_diagnostics.csv", index=False)
display(generation_analysis.groupby(["model", "temperature"])[
    ["unique_token_ratio", "repeated_bigram_ratio"]
].mean().reset_index())

## 10. Qualitative analysis and conclusion

Complete this section after inspecting the generated samples and metric tables. Do not select a winner based on one metric alone.

Suggested comparison questions:
1. Which model achieved the lowest test loss and perplexity?
2. Which model achieved the highest next-token accuracy?
3. Which model trained fastest?
4. At temperature 0.5, did outputs become more predictable or repetitive?
5. At temperature 1.0, did outputs balance fluency and variety?
6. At temperature 1.3, did diversity improve at the cost of coherence?
7. Which model best balances quantitative metrics and qualitative generation?

The best model should be justified from the **observed results**. A gated architecture may perform well, but do not assume LSTM or GRU wins before looking at your experiment.

In [ ]:
# Create a short machine-generated results summary from the actual measured metrics.
best_loss = results.loc[results["test_loss"].idxmin()]
best_acc = results.loc[results["test_accuracy"].idxmax()]
fastest = results.loc[results["training_time_seconds"].idxmin()]
summary = (
    f"Lowest test loss: {best_loss['model']} ({best_loss['test_loss']:.4f}).\\n"
    f"Highest test accuracy: {best_acc['model']} ({best_acc['test_accuracy']:.4f}).\\n"
    f"Fastest training: {fastest['model']} ({fastest['training_time_seconds']:.1f} seconds).\\n"
    "Use the generated samples and diagnostics to justify the overall best model."
)
print(summary)
(OUTPUT_DIR / "automatic_results_summary.txt").write_text(summary, encoding="utf-8")
print("Saved outputs to:", OUTPUT_DIR.resolve())

# Bundle the generated experiment outputs and trigger a download automatically in Google Colab.
import zipfile
RESULTS_ZIP = PROJECT_DIR / "ICT4403_Assignment2_Results.zip"
with zipfile.ZipFile(RESULTS_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(PROJECT_DIR))
print("Results archive created:", RESULTS_ZIP)
try:
    from google.colab import files as colab_files
    colab_files.download(str(RESULTS_ZIP))
except ImportError:
    print("Not running in Google Colab. Download this file manually:", RESULTS_ZIP)
